# 06 Multivariable Analysis — Exercises

Use the Pine and Cypress Nursing Home Legionnaires' disease line list to practice Modified Poisson regression (adjusted RR)
and logistic regression (adjusted OR), and compare the two.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# -- CJK font setup --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
fs_map = {"bedridden": 0, "assisted": 1, "independent": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

## Question 1: Predicting death—Crude RR vs Crude OR

Switch to a different outcome variable—predict **death** instead (`outcome == 'dead'`).

1. Build a `dead` column (0/1)
2. Compute the case fatality rate and decide: is it high or low? Will the OR and RR differ much?
3. For the following variables, compute the crude RR (Modified Poisson) and crude OR (logistic) **at the same time**:
   `age`, `comorbidity_chf`, `comorbidity_copd`, `immunosuppressed`,
   `clinical_severity` (as numbers: mild=1, moderate=2, severe=3)
4. Organize the results into a table and compare the RR and OR. When the fatality rate is lower, does the gap shrink?

In [ ]:
# TODO: build the dead column
# TODO: compute the case fatality rate
# TODO: loop to compute crude RR (Modified Poisson) and crude OR (logistic) at the same time
# TODO: organize into an RR vs OR comparison table

## Question 2: Multivariable Adjusted RR + Adjusted OR

Build a multivariable model predicting death:

```
dead ~ age + comorbidity_chf + comorbidity_copd + immunosuppressed + severity_score
```

1. Use **Modified Poisson** (`smf.glm(..., family=Poisson()).fit(cov_type='HC0')`) to compute the adjusted RR
2. Use **Logistic Regression** (`smf.logit()`) to compute the adjusted OR
3. Compare the adjusted RR and adjusted OR side by side—which variables differ the most?
4. Compare with the crude results from Question 1—which variable changes the most from crude → adjusted? (→ the most confounded factor)

In [ ]:
# TODO: Modified Poisson multivariable model → adjusted RR
# TODO: Logistic multivariable model → adjusted OR
# TODO: side-by-side comparison table
# TODO: compare crude vs adjusted

## Question 3 (challenge): Model comparison + Forest Plot

1. Build two Modified Poisson models:
   - Model A: `dead ~ age + immunosuppressed + severity_score`
   - Model B: `dead ~ age + comorbidity_chf + comorbidity_copd + immunosuppressed + severity_score`
2. Compare the AIC—which is better?
3. Using the better model, draw an **Adjusted RR** forest plot (refer to the forest plot code in the class notes)
4. Interpret: which factors are independent predictors of death (RR > 1 and the CI does not include 1)?

In [ ]:
# TODO: build models A and B (Modified Poisson + HC0)
# TODO: compare AIC
# TODO: Forest plot (adjusted RR)
# TODO: interpret

## Question 4: Risk Factors Among Tuberculosis Contacts (TB Scenario)

A community conducted contact investigation and chest X-ray screening for contacts of tuberculosis patients, enrolling 600 people and recording whether they were confirmed to have active pulmonary tuberculosis (`active_tb`).

1. Compute the prevalence of active tuberculosis in the sample.
2. Compute the crude OR and 95% confidence interval for "household close contact" (`close_contact`).
3. Build a multivariable logistic regression model:

   ```
   active_tb ~ age + diabetes + close_contact + underweight + smoking
   ```

   Compute the adjusted OR and 95% CI for each variable.
4. Compare the crude OR and adjusted OR for `close_contact` — is it affected by other confounders?
5. Interpret: after adjustment, which risk factors have a 95% CI that excludes 1 (statistically significant)? Which has the largest adjusted OR?

In [ ]:
# --- Data: TB contact investigation (synthetic data) ---
rng = np.random.default_rng(406)
n = 600

age = rng.normal(45, 15, n).clip(5, 90)
diabetes = (rng.uniform(0, 1, n) < (0.05 + age / 300)).astype(int)
close_contact = rng.binomial(1, 0.4, n)          # whether a cohabiting household close contact
underweight = rng.binomial(1, 0.15, n)           # BMI < 18.5 (underweight/malnourished)
smoking = rng.binomial(1, 0.25, n)

logit_p = (
    -4.3
    + 0.03 * age
    + 0.8 * diabetes
    + 1.3 * close_contact
    + 0.9 * underweight
    + 0.5 * smoking
)
p_tb = 1 / (1 + np.exp(-logit_p))
active_tb = rng.binomial(1, p_tb)

tb = pd.DataFrame({
    "age": age.round(1),
    "diabetes": diabetes,
    "close_contact": close_contact,
    "underweight": underweight,
    "smoking": smoking,
    "active_tb": active_tb,
})

# TODO: compute the prevalence of active_tb
# TODO: use smf.logit("active_tb ~ close_contact", data=tb) to compute the crude OR and 95% CI for close_contact
# TODO: build the multivariable model active_tb ~ age + diabetes + close_contact + underweight + smoking, and compute the adjusted OR
# TODO: compare the crude OR and adjusted OR for close_contact
# TODO: identify which variables have a 95% CI that excludes 1, and point out the risk factor with the largest adjusted OR

## Question 5: Predicting Severe COVID-19 (COVID-19 Scenario)

A hospital built a line list of confirmed COVID-19 cases from a community screening station, with 600 records, recording whether each case became severe (`severe`, requiring hospitalization or ICU admission).

1. Compute the proportion of severe cases.
2. Compute the crude OR and 95% CI for "unvaccinated" (`unvaccinated`).
3. Build a multivariable logistic regression model:

   ```
   severe ~ age + obesity + unvaccinated + chronic_lung
   ```

   Compute the adjusted OR and 95% CI for each variable.
4. Compare the crude OR and adjusted OR for `unvaccinated`.
5. Interpret: is the adjusted OR for `unvaccinated` significantly greater than 1 (95% CI excludes 1)? What does this imply for the conclusion that "vaccination reduces the risk of severe disease"?

In [ ]:
# --- Data: COVID-19 confirmed cases from community screening (synthetic data) ---
rng = np.random.default_rng(507)
n = 600

age = rng.normal(50, 18, n).clip(18, 95)
obesity = rng.binomial(1, 0.30, n)
unvaccinated = rng.binomial(1, 0.35, n)
chronic_lung = rng.binomial(1, 0.12, n)

logit_p = (
    -5.3
    + 0.05 * age
    + 0.8 * obesity
    + 1.1 * unvaccinated
    + 0.9 * chronic_lung
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe = rng.binomial(1, p_severe)

covid = pd.DataFrame({
    "age": age.round(1),
    "obesity": obesity,
    "unvaccinated": unvaccinated,
    "chronic_lung": chronic_lung,
    "severe": severe,
})

# TODO: compute the proportion of severe cases
# TODO: compute the crude OR and 95% CI for unvaccinated
# TODO: build the multivariable model severe ~ age + obesity + unvaccinated + chronic_lung, and compute the adjusted OR
# TODO: compare the crude OR and adjusted OR
# TODO: interpret whether the adjusted OR for unvaccinated is significantly greater than 1

## Question 6: Risk Factors for Severe Dengue (DHF) (Dengue Scenario)

During a dengue outbreak in a county, 550 confirmed cases were enrolled, recording whether each case developed severe dengue (`severe_dengue`, i.e., dengue hemorrhagic fever (DHF) / dengue shock syndrome (DSS)).

1. Compute the proportion of severe dengue cases.
2. Compute the crude OR and 95% CI for "secondary infection" (`secondary_infection`, having previously been infected with a different dengue serotype).
3. Build a multivariable logistic regression model:

   ```
   severe_dengue ~ secondary_infection + age + diabetes + hypertension
   ```

   Compute the adjusted OR and 95% CI for each variable.
4. Compare the crude OR and adjusted OR for `secondary_infection` — is it confounded by age or comorbidities?
5. Interpret: based on the adjusted OR, how many times higher are the odds of severe dengue for secondary-infection cases compared to primary-infection cases? Is this consistent with the pathological mechanism of antibody-dependent enhancement (ADE)?

In [ ]:
# --- Data: confirmed dengue cases (synthetic data) ---
rng = np.random.default_rng(608)
n = 550

secondary_infection = rng.binomial(1, 0.30, n)   # whether this is a secondary infection (different serotype)
age = rng.normal(35, 20, n).clip(1, 85)
diabetes = rng.binomial(1, 0.15, n)
hypertension = rng.binomial(1, 0.20, n)

logit_p = (
    -3.8
    + 1.5 * secondary_infection
    + 0.03 * age
    + 0.7 * diabetes
    + 0.5 * hypertension
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe_dengue = rng.binomial(1, p_severe)

dengue = pd.DataFrame({
    "secondary_infection": secondary_infection,
    "age": age.round(1),
    "diabetes": diabetes,
    "hypertension": hypertension,
    "severe_dengue": severe_dengue,
})

# TODO: compute the proportion of severe dengue cases
# TODO: compute the crude OR and 95% CI for secondary_infection
# TODO: build the multivariable model severe_dengue ~ secondary_infection + age + diabetes + hypertension, and compute the adjusted OR
# TODO: compare the crude OR and adjusted OR, and assess whether it is confounded
# TODO: interpret the meaning of adjusted OR(secondary_infection)

## Question 7: Risk Factors for Influenza Hospitalization (Influenza Scenario)

During a flu season, a community clinic built a line list of confirmed cases, with 700 records, recording whether each case was hospitalized after diagnosis (`hospitalized`).

1. Compute the proportion hospitalized.
2. Compute the crude OR and 95% CI for "vaccinated against influenza" (`vaccinated`) (hint: patients with chronic disease are often preferentially recommended for vaccination, so the crude OR may be distorted by confounding).
3. Build a multivariable logistic regression model:

   ```
   hospitalized ~ age + chronic_disease + vaccinated + late_treatment
   ```

   Compute the adjusted OR and 95% CI for each variable.
4. Compare the crude OR and adjusted OR for `vaccinated` — does the direction change after adjustment (e.g., from "increased risk" to "protective effect")?
5. Interpret: what is this phenomenon called, where "high-risk groups more often receive an intervention" distorts the direction of the association? Why does the association between vaccination and chronic disease history produce this pattern of confounding?

In [ ]:
# --- Data: confirmed influenza cases (synthetic data, includes confounding by indication) ---
rng = np.random.default_rng(709)
n = 700

age = rng.normal(40, 20, n).clip(0, 95)
chronic_disease = rng.binomial(1, 0.22, n)
# Patients with chronic disease are often preferentially recommended for vaccination -> vaccinated is strongly correlated with chronic_disease (confounding by indication)
p_vaccinated = 0.20 + 0.65 * chronic_disease
vaccinated = rng.binomial(1, p_vaccinated)
late_treatment = rng.binomial(1, 0.40, n)   # treatment started more than 48 hours after symptom onset

logit_p = (
    -4.1
    + 0.03 * age
    + 1.8 * chronic_disease
    - 0.65 * vaccinated
    + 0.9 * late_treatment
)
p_hosp = 1 / (1 + np.exp(-logit_p))
hospitalized = rng.binomial(1, p_hosp)

flu = pd.DataFrame({
    "age": age.round(1),
    "chronic_disease": chronic_disease,
    "vaccinated": vaccinated,
    "late_treatment": late_treatment,
    "hospitalized": hospitalized,
})

# TODO: compute the proportion hospitalized
# TODO: compute the crude OR and 95% CI for vaccinated
# TODO: build the multivariable model hospitalized ~ age + chronic_disease + vaccinated + late_treatment, and compute the adjusted OR
# TODO: compare the crude OR and adjusted OR, and check whether the direction changes
# TODO: interpret this confounding phenomenon (confounding by indication)

## Question 8 (Challenge): Interaction in Severe Measles Complications (Measles Scenario)

During a measles outbreak, 650 confirmed cases (mostly children) were enrolled, recording whether each developed a severe complication (`severe_complication`, e.g., pneumonia or encephalitis). Malnutrition and vitamin A deficiency are known to potentially interact synergistically through shared pathological mechanisms.

1. Split cases into four groups by `malnutrition` and `vitamin_a_deficiency` (neither / malnutrition only / vitamin A deficiency only / both), and compare the proportion of severe complications across groups to look for signs of a synergistic effect.
2. Build a main-effects model without an interaction term:

   ```
   severe_complication ~ malnutrition + vitamin_a_deficiency + age
   ```

   Compute the adjusted OR.
3. Build a model that includes the interaction term:

   ```
   severe_complication ~ malnutrition * vitamin_a_deficiency + age
   ```

   (statsmodels will automatically expand this to `malnutrition + vitamin_a_deficiency + malnutrition:vitamin_a_deficiency`); compute the OR and 95% CI for the interaction term.
4. Compare the two models using AIC — does the interaction term improve model fit?
5. Interpret: if the interaction term's OR is greater than 1, what does that mean in a logistic regression (a synergistic effect on the multiplicative scale)? What are the implications for public health policy (e.g., prioritizing vitamin A supplementation for malnourished children)?

In [ ]:
# --- Data: measles outbreak cases (synthetic data, includes malnutrition x vitamin_a_deficiency interaction) ---
rng = np.random.default_rng(810)
n = 650

age = rng.uniform(0.5, 15, n)                 # mostly children
malnutrition = rng.binomial(1, 0.25, n)
vitamin_a_deficiency = rng.binomial(1, 0.20, n)

logit_p = (
    -2.8
    + 0.6 * malnutrition
    + 0.6 * vitamin_a_deficiency
    + 1.6 * malnutrition * vitamin_a_deficiency   # interaction: risk is amplified when both are present
    - 0.05 * age
)
p_severe = 1 / (1 + np.exp(-logit_p))
severe_complication = rng.binomial(1, p_severe)

measles = pd.DataFrame({
    "age": age.round(2),
    "malnutrition": malnutrition,
    "vitamin_a_deficiency": vitamin_a_deficiency,
    "severe_complication": severe_complication,
})

# TODO: split into four groups (neither/malnutrition only/vitamin A deficiency only/both), and compare the proportion of severe cases across groups
# TODO: build the main-effects model severe_complication ~ malnutrition + vitamin_a_deficiency + age
# TODO: build the interaction model severe_complication ~ malnutrition * vitamin_a_deficiency + age
# TODO: compare the AIC of the two models
# TODO: interpret the OR of the interaction term and its public health implications